**Procesar precios de mienrales**

En este cuaderno estructuro la información de los siguientes datos para manejarlos en STATA:
- Precios de minerales armonizados (Fuente: World Bank - PinkSheet)

# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *



Setup general cargado


# Definir datos de salida
El archivo de salida se estructura para conservar la misma estructura entre todas las bases de datos

In [2]:
ORDEN_DF

['codigo_dane_municipio',
 'anno',
 'nombre_variable',
 'variable_sujeto',
 'variable_medicion',
 'variable_detalle',
 'variable_descripcion',
 'valor',
 'clasificacion_econometria']

In [3]:
# Definir DF con la estructura acordada para el proyecto
# La variable global ORDEN_DF ya tiene la lista de todas las columnas ordenadas
# Inicializar el DF vacío
df_salida = pd.DataFrame(
    columns=[ORDEN_DF]
)

# Precios minerales
Fuente: World Bank - Pink Sheet

In [6]:
# Cargar datos
precios_minerales_armonizados = pd.read_parquet(
    DATA/"intermediate/e2011_WbPinkSheet_preciosMinerales_armonizado.parquet"
)

print(MSC_SEPARADOR, "Head")
display(precios_minerales_armonizados.head())


-------------------------------- Head


,periodo,variable,value,texto_busqueda,categoria_armonizada
0,1970M01,"Coal, Australian ($/mt)",7.80,coal australian mt,Carbon
1,1970M02,"Coal, Australian ($/mt)",7.80,coal australian mt,Carbon
2,1970M03,"Coal, Australian ($/mt)",7.80,coal australian mt,Carbon
3,1970M04,"Coal, Australian ($/mt)",7.80,coal australian mt,Carbon
4,1970M05,"Coal, Australian ($/mt)",7.80,coal australian mt,Carbon


## Procesar las fechas

In [15]:
# Traducir de string a fecha
precios_minerales_armonizados["fecha_mes"] = pd.to_datetime(
    precios_minerales_armonizados["periodo"],
    format="%YM%m"
)

# Crear columna de año para agregar
precios_minerales_armonizados[COL_ANNO] = precios_minerales_armonizados['fecha_mes'].dt.year

## Agregar valores anuales
Pensando en probar hipotesis de que precios altos o bajos pueden generar auges/declives locales, para cada año me interesa tanto el promedio como los valores extremos

In [16]:
# Agregar los valores anualmente
precios_minerales_anuales = (
    precios_minerales_armonizados.groupby(['variable', COL_ANNO], as_index=False)
    .agg(
        precio_promedio=("value", "mean"),
        precio_minimo=("value", "min"),
        precio_maximo=("value", "max"),
        meses_disponibles=("value", "count"),
        categoria_armonizada=("categoria_armonizada", 'first')
    )
)

# Mostrar los nuevos datos
print(MSC_SEPARADOR, "DF")
display(precios_minerales_anuales)


-------------------------------- DF


,variable,anno,precio_promedio,precio_minimo,precio_maximo,meses_disponibles,categoria_armonizada
0,Aluminum ($/mt),1960,511.00,511.00,511.00,12,Metales base
1,Aluminum ($/mt),1961,511.00,511.00,511.00,12,Metales base
2,Aluminum ($/mt),1962,498.00,496.00,511.00,12,Metales base
3,Aluminum ($/mt),1963,498.75,496.00,507.00,12,Metales base
4,Aluminum ($/mt),1964,525.92,507.00,540.00,12,Metales base
...,...,...,...,...,...,...,...
966,Zinc ($/mt),2022,"3,481.25","2,939.00","4,360.00",12,Metales base
967,Zinc ($/mt),2023,"2,652.75","2,375.00","3,310.00",12,Metales base
968,Zinc ($/mt),2024,"2,775.75","2,360.00","3,106.00",12,Metales base
969,Zinc ($/mt),2025,"2,867.58","2,622.00","3,177.00",12,Metales base


## Estructurar datos de salida

In [58]:
# Estructurar los datos en el formato acordado
precios_minerales_salida = precios_minerales_anuales.melt(
    id_vars=[COL_ANNO, 'variable', 'categoria_armonizada'],
    value_name=COL_VALOR,
    var_name=COL_VARIABLE_MEDICION
)

# Asiganr nombres de variables acordados
precios_minerales_salida = precios_minerales_salida.rename(
    columns={
        'categoria_armonizada':COL_VARIABLE_SUJETO,
        'variable': COL_VARIABLE_DETALLE,
    }
)

# Completar información de columnas
precios_minerales_salida[COL_VARIABLE_DESCRIPCION]= (
    'Precio minerales. ' +
    precios_minerales_salida[COL_VARIABLE_SUJETO] + '. ' +
    precios_minerales_salida[COL_VARIABLE_DETALLE] + '. ' +
    precios_minerales_salida[COL_VARIABLE_MEDICION] + '. '
)
precios_minerales_salida[COL_CLASIFICACION_ECONOMETRIA] ='Instrumento: Precio'
precios_minerales_salida[COL_ID_MUNICIPIO] = None
precios_minerales_salida[COL_NOMBRE_DE_VARIABLE] = ''


In [63]:
print(MSC_SEPARADOR, "Explorar nombres y unidades de medicion")
display(precios_minerales_salida[COL_VARIABLE_DETALLE].unique())

diccionario_abreviaciones_metales_minerales = {
    'Aluminum ($/mt)': 'Al',
    'Coal, Australian ($/mt)': 'C_AU',
    'Coal, South African ** ($/mt)': 'C_SA',
    'Copper ($/mt)': 'Cu',
    'Gold ($/troy oz)': 'oro',
    'Iron ore, cfr spot ($/dmtu)': 'Fe',
    'Lead ($/mt)': 'Pb',
    'Nickel ($/mt)': 'Ni',
    'Phosphate rock ($/mt)': 'phsp',
    'Platinum ($/troy oz)': 'plat',
    'Potassium chloride ** ($/mt)': 'KCl',
    'Silver ($/troy oz)': 'Ag',
    'Tin ($/mt)': 'Sn',
    'Urea  ($/mt)': 'urea',
    'Zinc ($/mt)': 'zinc'
}

diccionario_unidades = {
    'Aluminum ($/mt)': 'USmt',
    'Coal, Australian ($/mt)': 'USmt',
    'Coal, South African ** ($/mt)': 'USmt',
    'Copper ($/mt)': 'USmt',
    'Gold ($/troy oz)': 'USoz',
    'Iron ore, cfr spot ($/dmtu)': 'USmt',
    'Lead ($/mt)': 'USmt',
    'Nickel ($/mt)': 'USmt',
    'Phosphate rock ($/mt)': 'USmt',
    'Platinum ($/troy oz)': 'USoz',
    'Potassium chloride ** ($/mt)': 'USmt',
    'Silver ($/troy oz)': 'USoz',
    'Tin ($/mt)': 'USmt',
    'Urea  ($/mt)': 'USmt',
    'Zinc ($/mt)': 'USmt'
}

print(MSC_SEPARADOR, "Explorar mediciones disponibles")
display(precios_minerales_salida[COL_VARIABLE_MEDICION].unique())


-------------------------------- Explorar nombres y unidades de medicion


array(['Aluminum ($/mt)', 'Coal, Australian ($/mt)',
       'Coal, South African ** ($/mt)', 'Copper ($/mt)',
       'Gold ($/troy oz)', 'Iron ore, cfr spot ($/dmtu)', 'Lead ($/mt)',
       'Nickel ($/mt)', 'Phosphate rock ($/mt)', 'Platinum ($/troy oz)',
       'Potassium chloride ** ($/mt)', 'Silver ($/troy oz)', 'Tin ($/mt)',
       'Urea  ($/mt)', 'Zinc ($/mt)'], dtype=object)


-------------------------------- Explorar mediciones disponibles


array(['precio_promedio', 'precio_minimo', 'precio_maximo',
       'meses_disponibles'], dtype=object)

In [73]:
# Asignar nombres de las variables con la secuencia definida
precios_minerales_salida[COL_NOMBRE_DE_VARIABLE]=(
    'precMine_' + 
    precios_minerales_salida[COL_VARIABLE_SUJETO].map(DICCIONARIO_ABREVIACIONES_MINERALES) + '_' +
    precios_minerales_salida['variable_medicion'].map({
        'precio_promedio':'prmdio', 'precio_minimo':'minimo', 'precio_maximo':'maximo', 'meses_disponibles':'conteo'}) + '_' +
    precios_minerales_salida[COL_VARIABLE_DETALLE].map(diccionario_abreviaciones_metales_minerales) + '_' +
    precios_minerales_salida[COL_VARIABLE_DETALLE].map(diccionario_unidades)
)

print(MSC_SEPARADOR, "Verificar que los nombres de las variables no superen los 32 caracteres")
display(precios_minerales_salida[COL_NOMBRE_DE_VARIABLE].str.len().max())


-------------------------------- Verificar que los nombres de las variables no superen los 32 caracteres


32

In [75]:
print(MSC_SEPARADOR, 'DF de salida')
display(precios_minerales_salida[ORDEN_DF])


-------------------------------- DF de salida


,codigo_dane_municipio,anno,nombre_variable,variable_sujeto,variable_medicion,variable_detalle,variable_descripcion,valor,clasificacion_econometria
0,None,1960,precMine_metBas_prmdio_Al_USmt,Metales base,precio_promedio,Aluminum ($/mt),Precio minerales. Metales base. Aluminum ($/mt...,511.00,Instrumento: Precio
1,None,1961,precMine_metBas_prmdio_Al_USmt,Metales base,precio_promedio,Aluminum ($/mt),Precio minerales. Metales base. Aluminum ($/mt...,511.00,Instrumento: Precio
2,None,1962,precMine_metBas_prmdio_Al_USmt,Metales base,precio_promedio,Aluminum ($/mt),Precio minerales. Metales base. Aluminum ($/mt...,498.00,Instrumento: Precio
3,None,1963,precMine_metBas_prmdio_Al_USmt,Metales base,precio_promedio,Aluminum ($/mt),Precio minerales. Metales base. Aluminum ($/mt...,498.75,Instrumento: Precio
4,None,1964,precMine_metBas_prmdio_Al_USmt,Metales base,precio_promedio,Aluminum ($/mt),Precio minerales. Metales base. Aluminum ($/mt...,525.92,Instrumento: Precio
...,...,...,...,...,...,...,...,...,...
3879,None,2022,precMine_metBas_conteo_zinc_USmt,Metales base,meses_disponibles,Zinc ($/mt),Precio minerales. Metales base. Zinc ($/mt). m...,12.00,Instrumento: Precio
3880,None,2023,precMine_metBas_conteo_zinc_USmt,Metales base,meses_disponibles,Zinc ($/mt),Precio minerales. Metales base. Zinc ($/mt). m...,12.00,Instrumento: Precio
3881,None,2024,precMine_metBas_conteo_zinc_USmt,Metales base,meses_disponibles,Zinc ($/mt),Precio minerales. Metales base. Zinc ($/mt). m...,12.00,Instrumento: Precio
3882,None,2025,precMine_metBas_conteo_zinc_USmt,Metales base,meses_disponibles,Zinc ($/mt),Precio minerales. Metales base. Zinc ($/mt). m...,12.00,Instrumento: Precio


## Preparar salida para STATA

In [80]:
# Los datos son precios internacionales, no corresponden a un municipio en particular.
# Por eso, la columna de municipio no tiene significado en este DF
precios_minerales_salida = precios_minerales_salida.drop(columns=COL_ID_MUNICIPIO)

In [81]:
# Asignar estructura wide para STATA
precios_minerales_salida_STATA = precios_minerales_salida.pivot(
    index=[COL_ANNO],
    columns=COL_NOMBRE_DE_VARIABLE,
    values=COL_VALOR
).reset_index().rename_axis(columns=None)



In [82]:
precios_minerales_salida_STATA

,anno,precMine_carbon_conteo_C_AU_USmt,precMine_carbon_conteo_C_SA_USmt,precMine_carbon_maximo_C_AU_USmt,precMine_carbon_maximo_C_SA_USmt,precMine_carbon_minimo_C_AU_USmt,precMine_carbon_minimo_C_SA_USmt,precMine_carbon_prmdio_C_AU_USmt,precMine_carbon_prmdio_C_SA_USmt,precMine_cobre_conteo_Cu_USmt,precMine_cobre_maximo_Cu_USmt,precMine_cobre_minimo_Cu_USmt,precMine_cobre_prmdio_Cu_USmt,precMine_fertlz_conteo_KCl_USmt,precMine_fertlz_conteo_phsp_USmt,precMine_fertlz_conteo_urea_USmt,precMine_fertlz_maximo_KCl_USmt,precMine_fertlz_maximo_phsp_USmt,precMine_fertlz_maximo_urea_USmt,precMine_fertlz_minimo_KCl_USmt,precMine_fertlz_minimo_phsp_USmt,precMine_fertlz_minimo_urea_USmt,precMine_fertlz_prmdio_KCl_USmt,precMine_fertlz_prmdio_phsp_USmt,precMine_fertlz_prmdio_urea_USmt,precMine_metBas_conteo_Al_USmt,precMine_metBas_conteo_Fe_USmt,precMine_metBas_conteo_Pb_USmt,precMine_metBas_conteo_Sn_USmt,precMine_metBas_conteo_zinc_USmt,precMine_metBas_maximo_Al_USmt,precMine_metBas_maximo_Fe_USmt,precMine_metBas_maximo_Pb_USmt,precMine_metBas_maximo_Sn_USmt,precMine_metBas_maximo_zinc_USmt,precMine_metBas_minimo_Al_USmt,precMine_metBas_minimo_Fe_USmt,precMine_metBas_minimo_Pb_USmt,precMine_metBas_minimo_Sn_USmt,precMine_metBas_minimo_zinc_USmt,precMine_metBas_prmdio_Al_USmt,precMine_metBas_prmdio_Fe_USmt,precMine_metBas_prmdio_Pb_USmt,precMine_metBas_prmdio_Sn_USmt,precMine_metBas_prmdio_zinc_USmt,precMine_niquel_conteo_Ni_USmt,precMine_niquel_maximo_Ni_USmt,precMine_niquel_minimo_Ni_USmt,precMine_niquel_prmdio_Ni_USmt,precMine_oPreci_conteo_Ag_USoz,precMine_oPreci_conteo_plat_USoz,precMine_oPreci_maximo_Ag_USoz,precMine_oPreci_maximo_plat_USoz,precMine_oPreci_minimo_Ag_USoz,precMine_oPreci_minimo_plat_USoz,precMine_oPreci_prmdio_Ag_USoz,precMine_oPreci_prmdio_plat_USoz,precMine_oro_conteo_oro_USoz,precMine_oro_maximo_oro_USoz,precMine_oro_minimo_oro_USoz,precMine_oro_prmdio_oro_USoz
0,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.00,728.00,613.00,678.75,12.00,12.00,12.00,28.50,13.00,42.30,28.50,13.00,42.30,28.50,13.00,42.30,12.00,12.00,12.00,12.00,12.00,511.00,11.40,214.00,"2,249.00",261.00,511.00,11.40,179.00,"2,163.00",228.00,511.00,11.40,198.75,"2,196.83",246.25,12.00,"1,631.00","1,631.00","1,631.00",12.00,12.00,0.90,84.00,0.90,84.00,0.90,84.00,12.00,35.00,35.00,35.00
1,1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.00,668.00,607.00,633.00,12.00,12.00,12.00,30.00,13.00,42.30,30.00,13.00,42.30,30.00,13.00,42.30,12.00,12.00,12.00,12.00,12.00,511.00,11.00,185.00,"2,668.00",233.00,511.00,11.00,166.00,"2,163.00",192.00,511.00,11.00,177.00,"2,450.00",214.33,12.00,"1,711.00","1,711.00","1,711.00",12.00,12.00,1.00,84.00,0.90,84.00,0.91,84.00,12.00,35.00,35.00,35.00
2,1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.00,648.00,635.00,645.42,12.00,12.00,12.00,30.00,11.50,42.30,30.00,11.50,42.30,30.00,11.50,42.30,12.00,12.00,12.00,12.00,12.00,511.00,11.00,167.00,"2,652.00",194.00,496.00,11.00,141.00,"2,350.00",177.00,498.00,11.00,155.33,"2,471.42",186.08,12.00,"1,761.00","1,761.00","1,761.00",12.00,12.00,1.20,84.00,1.00,84.00,1.07,84.00,12.00,35.00,35.00,35.00
3,1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.00,650.00,646.00,646.33,12.00,12.00,12.00,30.00,11.50,42.30,30.00,11.50,42.30,30.00,11.50,42.30,12.00,12.00,12.00,12.00,12.00,507.00,11.00,205.00,"2,782.00",261.00,496.00,11.00,149.00,"2,350.00",186.00,498.75,11.00,174.92,"2,507.67",211.50,12.00,"1,742.00","1,741.00","1,741.67",12.00,12.00,1.30,84.00,1.20,79.00,1.29,81.33,12.00,35.00,35.00,35.00
4,1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.00,"1,400.00",656.00,969.67,12.00,12.00,12.00,32.50,12.50,60.50,32.50,12.50,60.50,32.50,12.50,60.50,12.00,12.00,12.00,12.00,12.00,540.00,10.20,384.00,"4,372.00",385.00,507.00,10.20,218.00,"2,873.00",264.00,525.92,10.20,278.00,"3,412.75",326.50,12.00,"1,742.00","1,741.00","1,741.67",12.00,12.00,1.30,89.00,1.30,88.00,1.30,88.92,12.00,35.00,35.00,35.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

## Exportar archivos

In [87]:
# Exportar ambos archivos
nombre_salida = 'e3000_precios_minerales'

precios_minerales_salida.to_parquet(DATA/f'intermediate/{nombre_salida}.parquet',
    index=False)
precios_minerales_salida_STATA.to_stata(DATA/f'intermediate/{nombre_salida}.dta',
    write_index=False)